# Eksperimen Preprocessing & Pemodelan Machine Learning
**Nama:** M Zulfan Afandi (jajangbakwan)
**Kelas:** Membangun Sistem Machine Learning - Proyek Akhir Dicoding
**Studi Kasus:** Prediksi MVP Pertandingan Mobile Legends M5 World Championship

## 1. Import Library
Mengimpor pustaka yang diperlukan untuk eksplorasi data, pra-pemrosesan, dan evaluasi model.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

print('Semua pustaka berhasil diimpor!')

## 2. Memuat Dataset Mentah (Raw Data)
Memuat berkas `Full Match Results Data.csv` dari folder dataset mentah.

In [2]:
raw_path = '../namadataset_raw/Full Match Results Data.csv'
if not os.path.exists(raw_path):
    raw_path = 'namadataset_raw/Full Match Results Data.csv'
df_raw = pd.read_csv(raw_path)
print(f'Dimensi data mentah: {df_raw.shape}')
df_raw.head()

## 3. Eksplorasi Data (EDA)
Memeriksa struktur kolom, tipe data, serta nilai yang hilang (missing values).

In [3]:
print('--- Informasi Dataset ---')
df_raw.info()

print('\n--- Pengecekan Missing Value ---')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

In [4]:
print('Distribusi Nilai Target MVP:')
print(df_raw['MVP'].value_counts())
df_raw['MVP'].value_counts().plot(kind='bar', title='Distribusi MVP')
plt.xlabel('Kelas MVP')
plt.ylabel('Jumlah')
plt.show()

## 4. Pembersihan Data (Data Cleaning)
Menghapus baris yang mengandung nilai kosong (missing values).

In [5]:
print(f'Jumlah baris sebelum cleaning: {len(df_raw)}')
df_clean = df_raw.dropna().copy()
print(f'Jumlah baris setelah cleaning: {len(df_clean)}')


## 5. Seleksi Fitur & Transformasi (Encoding)
Memilih fitur yang relevan: `Team`, `Ban`, `Role Ban`, `Pick`, `Role Pick` dan target `MVP`.
Menerapkan Label Encoding agar data siap diproses algoritma machine learning.

In [6]:
features = ['Team', 'Ban', 'Role Ban', 'Pick', 'Role Pick']
target = 'MVP'
df_selected = df_clean[features + [target]].copy()

encoders = {}
for col in features:
    le = LabelEncoder()
    df_selected[col] = le.fit_transform(df_selected[col].astype(str))
    encoders[col.lower().replace(' ', '_')] = le

df_preprocessed = df_selected.rename(columns={
    'Team': 'team',
    'Ban': 'ban',
    'Role Ban': 'role_ban',
    'Pick': 'pick',
    'Role Pick': 'role_pick',
    'MVP': 'mvp'
})

print('Dataframe setelah transformasi:')
print(df_preprocessed.head())
print('\nDimensi preprocessed data:', df_preprocessed.shape)


## 6. Uji Coba Model Baseline
Melakukan train-test split dan evaluasi sederhana dengan RandomForestClassifier.

In [7]:
X = df_preprocessed.drop(columns=['mvp'])
y = df_preprocessed['mvp']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Akurasi Model Baseline: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))


## 7. Menyimpan Dataset Hasil Preprocessing
Menyimpan dataset hasil preprocessing ke berkas `namadataset_preprocessing.csv`.

In [8]:
output_csv = 'namadataset_preprocessing.csv'
df_preprocessed.to_csv(output_csv, index=False)
joblib.dump(encoders, 'encoder_mappings.pkl')
print(f'Dataset preprocessing berhasil disimpan ke {output_csv}')
print('Label encoders berhasil disimpan ke encoder_mappings.pkl')
